In [ ]:
import numpy as np
import pandas as pd
import optuna
from catboost import CatBoostRegressor
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import root_mean_squared_error

warnings.simplefilter("ignore")

In [38]:
df = pd.read_csv("train.csv")

X = df.drop(columns=["project_id", "hours_spent", "log_hours", "hours_bucket"])
y = df["hours_spent"]
y_log = np.log1p(y)

print(f"Rows: {len(df):,}")
print(f"Features: {X.shape[1]}")

Rows: 4,500
Features: 69


In [39]:
PRIOR_MAP = {"0": 0, "1-2": 1, "3-5": 2, "6+": 3}
AGE_MAP = {"<1": 0, "1-2": 1, "3-5": 2, "6+": 3}
BOOL_MAP = {True: 1.0, False: 0.0, "True": 1.0, "False": 0.0}

CAT_COLS = [
    "primary_language",
    "framework",
    "hosting_type",
    "commit_cadence",
    "project_category",
]

BOOL_COLS = [
    "uses_game_engine", "has_tests", "has_ci_config", "has_dockerfile",
    "has_backend", "has_database", "has_custom_art",
    "readme_has_install_instructions", "has_license", "has_docs_folder",
    "url_is_live", "is_mobile_responsive", "has_custom_domain",
    "is_multiplayer", "mentions_ai_api", "is_clone_or_tutorial",
    "is_forked_from_template",
]


def clean_features(X):
    X = X.copy()
    X["author_prior_projects"] = X["author_prior_projects"].astype(str).map(PRIOR_MAP)
    X["author_account_age_years"] = X["author_account_age_years"].astype(str).map(AGE_MAP)

    for column in BOOL_COLS:
        if column in X.columns:
            X[column] = X[column].map(BOOL_MAP)

    for column in CAT_COLS:
        X[column] = X[column].fillna("missing").astype(str)

    return X


def divide(numerator, denominator):
    return numerator / denominator.replace(0, np.nan)


def add_features(X):
    X = clean_features(X)
    X = X.copy()

    X["lines_per_active_day"] = divide(X.lines_added_total, X.active_commit_days)
    X["loc_per_commit"] = divide(X.lines_of_code, X.num_commits)
    X["commits_per_active_day"] = divide(X.num_commits, X.active_commit_days)
    X["active_frac"] = divide(X.active_commit_days, X.project_span_days + 1)
    X["total_lines_touched"] = np.log1p(X.lines_added_total + X.lines_deleted_total)
    X["biggest_commit_share"] = divide(X.max_lines_in_one_commit, X.lines_added_total)
    X["bulk_upload"] = (X.pct_code_in_first_commit > 0.8).astype(float).where(
        X.pct_code_in_first_commit.notna()
    )
    X["few_commits"] = (X.num_commits <= 3).astype(float).where(X.num_commits.notna())
    X["no_git"] = X.num_commits.isna().astype(float)
    X["n_assets"] = X[[
        "num_image_files", "num_audio_files", "num_3d_model_files", "num_font_files"
    ]].sum(axis=1, min_count=1)
    X["readme_richness"] = X[[
        "readme_num_headings", "readme_num_images", "readme_num_code_blocks"
    ]].sum(axis=1, min_count=1)
    X["loc_x_days"] = np.log1p(X.lines_of_code) * np.log1p(X.active_commit_days)
    X["loc_template_adj"] = (
        np.log1p(X.lines_of_code)
        * (1 - 0.5 * X.is_forked_from_template.fillna(0))
    )

    return X


Xf = add_features(X)

In [ ]:
MODEL_PARAMS = {
    "cat_features": CAT_COLS,
    "iterations": 1245,
    "learning_rate": 0.03,
    "depth": 8,
    "l2_leaf_reg": 3,
    "rsm": 0.5,
    "loss_function": "RMSE",
    "random_seed": 42,
    "allow_writing_files": False,
    "verbose": 0,
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
oof_predictions = np.zeros(len(Xf))
fold_scores = []

for train_idx, val_idx in cv.split(Xf, df["hours_bucket"]):
    model = CatBoostRegressor(**MODEL_PARAMS)
    model.fit(Xf.iloc[train_idx], y_log.iloc[train_idx])

    oof_predictions[val_idx] = model.predict(Xf.iloc[val_idx])
    score = root_mean_squared_error(y_log.iloc[val_idx], oof_predictions[val_idx])
    fold_scores.append(score)

print("Fold scores:", ", ".join(f"{score:.5f}" for score in fold_scores))
print(f"Mean CV score: {np.mean(fold_scores):.5f} +/- {np.std(fold_scores):.5f}")
print(f"OOF score: {root_mean_squared_error(y_log, oof_predictions):.5f}")

Fold scores: 0.64933, 0.67595, 0.65043, 0.67315, 0.65277
Mean CV score: 0.66033 +/- 0.01170
OOF score: 0.66043


In [41]:
final_model = CatBoostRegressor(**MODEL_PARAMS)
final_model.fit(Xf, y_log)
final_model.save_model("model.cbm")

importance = final_model.get_feature_importance(prettified=True)
print(importance.head(25).to_string(index=False))

              Feature Id  Importances
              loc_x_days         6.13
  description_word_count         6.00
      active_commit_days         4.52
 median_lines_per_commit         3.42
        primary_language         3.13
        loc_template_adj         3.06
 pct_generic_commit_msgs         2.66
        project_category         2.65
            hosting_type         2.58
        num_source_files         2.40
pct_code_in_first_commit         2.35
     pct_commits_weekend         2.30
    biggest_commit_share         2.23
       lines_added_total         2.18
       readme_word_count         2.17
             active_frac         2.16
  commits_per_active_day         2.03
    pct_primary_language         2.02
  pct_commits_late_night         1.90
   avg_commit_msg_length         1.90
       project_span_days         1.85
      comment_line_ratio         1.81
     total_lines_touched         1.81
             num_commits         1.77
         avg_file_length         1.75


In [42]:
residuals = oof_predictions - y_log
print(pd.DataFrame({
    "hours_bucket": df["hours_bucket"],
    "residual_log_hours": residuals,
}).groupby("hours_bucket")["residual_log_hours"].agg(["mean", "std", "count"]))

              mean  std  count
hours_bucket                  
10-30        -0.30 0.50   1120
100+         -1.66 0.58     63
30-100       -0.89 0.55    332
<10           0.23 0.54   2985
